# 1. Ingesta, curaduría y calidad de datos

**Propósito.** Convertir la fuente experimental original en una tabla analítica trazable y verificar su calidad antes del EDA. Este cuaderno **sí está ejecutado**: cada celda numerada contiene su salida.

| Entrada | Controles | Salidas |
|---|---|---|
| `data/raw/dati_campagna_venditti.m` | hash, esquema, tipos, faltantes, duplicados, constantes, exclusiones y variables posensayo | tabla intermedia completa, tabla de modelado y reporte de calidad |

La curaduría no estima parámetros del modelo ni elimina observaciones por ser atípicas. Tampoco imputa la carga de primera falla.

In [1]:
from pathlib import Path
import json, shutil, sys
import pandas as pd
import yaml
from IPython.display import Markdown, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "ingesta.py").is_file()
             and (p / "config" / "data_schema.yml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto")
sys.path.insert(0, str(ROOT))

from src.ingesta import ingest, load_table, sha256_file
from src.preprocesamiento import curate_file

RAW_FILE = ROOT / "data" / "raw" / "dati_campagna_venditti.m"
SCHEMA_FILE = ROOT / "config" / "data_schema.yml"
assert RAW_FILE.is_file(), f"Fuente no encontrada: {RAW_FILE}"

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
print(f"Raíz del proyecto: {ROOT}")
print(f"Fuente encontrada: {RAW_FILE.relative_to(ROOT)}")
print(f"SHA-256 original: {sha256_file(RAW_FILE)}")

Raíz del proyecto: C:\Users\MELISSA\Downloads\pantographic-first-failure\pantographic-first-failure\T02_MIA14_SeccionB
Fuente encontrada: data\raw\dati_campagna_venditti.m
SHA-256 original: 3d82f876bb8ec652184707235aa4df1dfb4b1a6615551f78fe0e0662a6723f0c


## 1.1 Ingesta trazable

El lector restringido extrae la tabla literal `samples` del archivo MATLAB **sin ejecutar código MATLAB**. La fuente en `raw` permanece inmutable; cada ejecución crea un directorio fechado en `data/interim/`.

In [2]:
source_df, manifest = ingest(RAW_FILE)
interim_path = ROOT / manifest["interim_file"]

summary_ingestion = pd.DataFrame({
    "control": ["archivo fuente", "formato", "filas leídas", "columnas leídas", "hash fuente", "salida versionada", "lector"],
    "resultado": [manifest["source_file"], manifest["reader"]["format"], manifest["rows"], manifest["columns"],
                  manifest["sha256"], manifest["interim_file"], manifest["reader"]["parser"]]
})
display(summary_ingestion)
display(source_df.head())

,control,resultado
0,archivo fuente,data/raw/dati_campagna_venditti.m
1,formato,.m
2,filas leídas,27
3,columnas leídas,20
4,hash fuente,3d82f876bb8ec652184707235aa4df1dfb4b1a6615551f78fe0e0662a6723f0c
5,salida versionada,data/interim/20261003T021320337207Z/dataset_interim.csv
6,lector,src.importar_matlab.load_matlab_samples


,Case_n,n_cells_Y,Fiber_height,Fiber_base,Fiber_total_length,Fiber_total_volume,Pivot_height,Pivot_radius,Pivot_total_number,Pivot_total_volume,Sample_total_volume,First_Failure_Load_N,Ultimate_Load,Maximum_Displacement,residual_Displacement_1cycle_10mm,residual_Displacement_2cycle_20mm,residual_Displacement_3cycle_30mm,residual_Displacement_4cycle_40mm,residual_Displacement_5cycle_50mm,residual_Displacement_6cycle_60mm
0,1,4.0,1.0,2.40,2685.82,6435.30,1.0,0.5,113.0,88.71,6249.74,47.55,51.19,72.48,5.97,11.28,17.17,24.10,29.89,NaN
1,2,4.0,1.2,2.00,2685.82,6435.30,1.0,0.5,113.0,88.71,6148.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,4.0,1.4,1.71,2685.82,6435.30,1.0,0.5,113.0,88.71,6122.21,88.50,73.79,74.28,6.47,11.32,17.70,24.63,32.63,41.28
3,4,4.0,1.0,2.39,2685.82,6417.56,1.2,0.5,113.0,106.45,6240.20,86.45,93.37,85.11,4.50,9.05,13.88,19.45,24.93,33.48
4,5,4.0,1.2,1.99,2685.82,6417.56,1.2,0.5,113.0,106.45,6170.23,59.68,76.33,66.97,4.92,10.15,16.16,21.92,30.16,NaN


## 1.2 Esquema y roles de las variables

El esquema declara qué entra al modelo antes de mirar resultados. El identificador conserva trazabilidad y las respuestas medidas después del ensayo quedan fuera de los predictores para evitar fuga de información.

In [3]:
schema = yaml.safe_load(SCHEMA_FILE.read_text(encoding="utf-8"))
roles = schema["columns"]
units = schema["predictor_units"]
rows = []
for column in manifest["column_names"]:
    if column == roles["target"]:
        role, unit, moment = "objetivo", schema["project"]["target_unit"], "durante el ensayo"
    elif column in roles["identifiers"]:
        role, unit, moment = "identificador", "—", "antes del ensayo"
    elif column in roles["predictors"]:
        role, unit, moment = "predictor", units[column], "antes del ensayo"
    else:
        role, unit, moment = "no seleccionada", "según fuente", "respuesta posensayo"
    rows.append((column, role, unit, moment))
data_dictionary = pd.DataFrame(rows, columns=["variable", "rol", "unidad", "disponibilidad"])
display(data_dictionary)

,variable,rol,unidad,disponibilidad
0,Case_n,identificador,—,antes del ensayo
1,n_cells_Y,predictor,conteo,antes del ensayo
2,Fiber_height,predictor,mm,antes del ensayo
3,Fiber_base,predictor,mm,antes del ensayo
4,Fiber_total_length,predictor,mm,antes del ensayo
5,Fiber_total_volume,predictor,mm^3,antes del ensayo
6,Pivot_height,predictor,mm,antes del ensayo
7,Pivot_radius,predictor,mm,antes del ensayo
8,Pivot_total_number,predictor,conteo,antes del ensayo
9,Pivot_total_volume,predictor,mm^3,antes del ensayo


## 1.3 Curaduría determinística

Se validan encabezados, tipos numéricos, integridad del manifiesto y reglas del esquema. Un registro sin objetivo no puede usarse en aprendizaje supervisado y queda documentado, sin inventar su valor.

In [4]:
quality = curate_file(interim_path, SCHEMA_FILE)
processed_path = ROOT / quality["output_file"]
model_df = pd.read_csv(processed_path)

# Copias preliminares de acceso simple; las versiones fechadas se conservan.
interim_prelim = ROOT / "data" / "interim" / "preliminary"
processed_prelim = ROOT / "data" / "processed" / "preliminary"
interim_prelim.mkdir(parents=True, exist_ok=True)
processed_prelim.mkdir(parents=True, exist_ok=True)
for src, dst in [
    (interim_path, interim_prelim / "dataset_interim.csv"),
    (interim_path.parent / "dataset_manifest.json", interim_prelim / "dataset_manifest.json"),
    (processed_path, processed_prelim / "model_table.csv"),
    (processed_path.parent / "data_quality_report.json", processed_prelim / "data_quality_report.json"),
]:
    shutil.copy2(src, dst)

print(f"Ejecución de ingesta: {manifest['run_id']}")
print(f"Ejecución de curaduría: {quality['run_id']}")
print(f"Tabla analítica: {quality['output_file']}")
print("Copias preliminary actualizadas.")
display(model_df.head())

Ejecución de ingesta: 20261003T021320337207Z
Ejecución de curaduría: 20261003T021320523691Z
Tabla analítica: data/processed/20261003T021320523691Z/model_table.csv
Copias preliminary actualizadas.


,__source_record,Case_n,n_cells_Y,Fiber_height,Fiber_base,Fiber_total_length,Fiber_total_volume,Pivot_height,Pivot_radius,Pivot_total_number,Pivot_total_volume,Sample_total_volume,First_Failure_Load_N
0,1,1,4.0,1.0,2.40,2685.82,6435.30,1.0,0.5,113.0,88.71,6249.74,47.55
1,3,3,4.0,1.4,1.71,2685.82,6435.30,1.0,0.5,113.0,88.71,6122.21,88.50
2,4,4,4.0,1.0,2.39,2685.82,6417.56,1.2,0.5,113.0,106.45,6240.20,86.45
3,5,5,4.0,1.2,1.99,2685.82,6417.56,1.2,0.5,113.0,106.45,6170.23,59.68
4,6,6,4.0,1.4,1.71,2685.82,6417.56,1.2,0.5,113.0,106.45,6139.96,63.05


## 1.4 Diagnóstico de calidad antes y después

La comparación permite ver qué se corrigió, qué se excluyó y qué debe resolverse científicamente. Una columna constante se conserva en la tabla para trazabilidad y se eliminará dentro de cada pipeline de entrenamiento.

In [5]:
before, after = quality["before"], quality["after"]
quality_overview = pd.DataFrame({
    "indicador": ["filas", "columnas", "celdas faltantes", "filas duplicadas", "columnas constantes", "identificadores repetidos"],
    "fuente/interim": [before["rows"], before["columns"], sum(before["missing_by_column"].values()),
                       before["duplicated_rows"], ", ".join(before["constant_columns"]) or "ninguna", "—"],
    "tabla analítica": [after["rows"], after["columns"], sum(after["missing_by_column"].values()),
                        after["duplicated_rows"], ", ".join(after["constant_columns"]) or "ninguna",
                        quality["rows_with_repeated_identifiers"]],
})
display(quality_overview)

missing = pd.DataFrame({
    "faltantes_fuente": pd.Series(before["missing_by_column"]),
    "faltantes_tabla_analítica": pd.Series(after["missing_by_column"]),
}).fillna("no seleccionada")
display(missing.loc[(missing["faltantes_fuente"] != 0) | (missing["faltantes_tabla_analítica"] != 0)])

,indicador,fuente/interim,tabla analítica
0,filas,27,26
1,columnas,20,12
2,celdas faltantes,15,0
3,filas duplicadas,0,0
4,columnas constantes,Pivot_radius,Pivot_radius
5,identificadores repetidos,—,0


,faltantes_fuente,faltantes_tabla_analítica
First_Failure_Load_N,1,0.0
Maximum_Displacement,1,no seleccionada
Ultimate_Load,1,no seleccionada
residual_Displacement_1cycle_10mm,1,no seleccionada
residual_Displacement_2cycle_20mm,1,no seleccionada
residual_Displacement_3cycle_30mm,1,no seleccionada
residual_Displacement_4cycle_40mm,1,no seleccionada
residual_Displacement_5cycle_50mm,1,no seleccionada
residual_Displacement_6cycle_60mm,7,no seleccionada


## 1.5 Exclusiones y decisiones de curaduría

Se distingue entre **excluir por imposibilidad de supervisión** y **no seleccionar por fuga temporal**. Ningún punto se elimina por su magnitud en esta etapa.

In [6]:
exclusions = pd.DataFrame(quality["excluded_records"])
if exclusions.empty:
    display(Markdown("**Exclusiones:** ninguna."))
else:
    exclusions["motivo_español"] = exclusions["reason"].map({"missing_target": "objetivo no medido"}).fillna(exclusions["reason"])
    display(exclusions)

decisions = pd.DataFrame({
    "decisión": ["Imputar objetivo", "Usar identificador como predictor", "Eliminar duplicados automáticamente",
                 "Excluir variables posensayo", "Eliminar atípicos automáticamente"],
    "aplicación": ["No", "No", "No", "Sí", "No"],
    "razón": ["alteraría la verdad experimental", "solo identifica el caso", "pueden ser réplicas válidas",
              "no estarían disponibles para predecir antes del ensayo", "requiere justificación física y análisis de sensibilidad"]
})
display(decisions)
print("Variables no seleccionadas:")
for item in quality["unselected_columns"]:
    print(" •", item)

,source_record,reason,motivo_español
0,2,missing_target,objetivo no medido


,decisión,aplicación,razón
0,Imputar objetivo,No,alteraría la verdad experimental
1,Usar identificador como predictor,No,solo identifica el caso
2,Eliminar duplicados automáticamente,No,pueden ser réplicas válidas
3,Excluir variables posensayo,Sí,no estarían disponibles para predecir antes del ensayo
4,Eliminar atípicos automáticamente,No,requiere justificación física y análisis de sensibilidad


Variables no seleccionadas:
 • Ultimate_Load
 • Maximum_Displacement
 • residual_Displacement_1cycle_10mm
 • residual_Displacement_2cycle_20mm
 • residual_Displacement_3cycle_30mm
 • residual_Displacement_4cycle_40mm
 • residual_Displacement_5cycle_50mm
 • residual_Displacement_6cycle_60mm


## 1.6 Evidencia de salidas e integridad

Los hashes permiten demostrar que el contenido revisado coincide con la ejecución registrada.

In [7]:
evidence = pd.DataFrame([
    ("raw", manifest["source_file"], manifest["sha256"], RAW_FILE.exists()),
    ("interim versionado", manifest["interim_file"], manifest["interim_sha256"], interim_path.exists()),
    ("processed versionado", quality["output_file"], quality["output_sha256"], processed_path.exists()),
    ("interim preliminary", "data/interim/preliminary/dataset_interim.csv", sha256_file(interim_prelim / "dataset_interim.csv"), True),
    ("processed preliminary", "data/processed/preliminary/model_table.csv", sha256_file(processed_prelim / "model_table.csv"), True),
], columns=["nivel", "ruta", "SHA-256", "existe"])
evidence["hash_verificado"] = [
    sha256_file(RAW_FILE) == manifest["sha256"],
    sha256_file(interim_path) == manifest["interim_sha256"],
    sha256_file(processed_path) == quality["output_sha256"],
    sha256_file(interim_prelim / "dataset_interim.csv") == manifest["interim_sha256"],
    sha256_file(processed_prelim / "model_table.csv") == quality["output_sha256"],
]
display(evidence)
assert evidence["hash_verificado"].all()
print("Integridad verificada: todos los hashes coinciden.")

,nivel,ruta,SHA-256,existe,hash_verificado
0,raw,data/raw/dati_campagna_venditti.m,3d82f876bb8ec652184707235aa4df1dfb4b1a6615551f78fe0e0662a6723f0c,True,True
1,interim versionado,data/interim/20261003T021320337207Z/dataset_interim.csv,220bd1022cf05e227ef029477ebc685df74f5034b9846790421da2c0019cf2b9,True,True
2,processed versionado,data/processed/20261003T021320523691Z/model_table.csv,2d0a17f54bdc6a6361aaa6e25a57cf67775c7d47a5e2adefb408f114defa21f8,True,True
3,interim preliminary,data/interim/preliminary/dataset_interim.csv,220bd1022cf05e227ef029477ebc685df74f5034b9846790421da2c0019cf2b9,True,True
4,processed preliminary,data/processed/preliminary/model_table.csv,2d0a17f54bdc6a6361aaa6e25a57cf67775c7d47a5e2adefb408f114defa21f8,True,True


Integridad verificada: todos los hashes coinciden.


## 1.7 Resultado y paso siguiente

La tabla de modelado contiene el identificador, los descriptores geométricos previos al ensayo y la **carga de primera falla en N**. El caso sin objetivo quedó registrado; las variables posensayo no entran al modelo; los faltantes del conjunto analítico son cero; `Pivot_radius` es constante y deberá tratarse dentro del pipeline.

El siguiente cuaderno es **[02_EDA_Avanzado.ipynb](02_EDA_Avanzado.ipynb)**. Allí se realizan los análisis univariado, bivariado y multivariado y la detección de valores atípicos.